In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.tree import DecisionTreeClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

file_path = '/Users/sandhyavenkataramaiah/Downloads/ai4i2020.csv'
data = pd.read_csv(file_path)

data_prepared = data.drop(['UDI', 'Product ID'], axis=1) 
X = data_prepared.drop('Machine failure', axis=1)
y = data_prepared['Machine failure']


categorical_cols = [cname for cname in X.columns if X[cname].dtype == "object"]
numerical_cols = [cname for cname in X.columns if X[cname].dtype in ['int64', 'float64']]


preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_cols)
    ])


X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0)
X_train = preprocessor.fit_transform(X_train)
X_test = preprocessor.transform(X_test)

def train_and_evaluate_model(model, X_train, y_train, X_test, y_test):
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    return accuracy, precision, recall, f1

models = {
    "Decision Tree": DecisionTreeClassifier(),
    "Neural Network (MLP)": MLPClassifier(max_iter=1000),
    "Support Vector Machine (SVM)": SVC()
}

for name, model in models.items():
    metrics = train_and_evaluate_model(model, X_train, y_train, X_test, y_test)
    print(f"{name} - Accuracy: {metrics[0]:.2f}, Precision: {metrics[1]:.2f}, Recall: {metrics[2]:.2f}, F1-Score: {metrics[3]:.2f}")

Decision Tree - Accuracy: 1.00, Precision: 0.99, Recall: 0.93, F1-Score: 0.96
Neural Network (MLP) - Accuracy: 1.00, Precision: 1.00, Recall: 0.93, F1-Score: 0.97
Support Vector Machine (SVM) - Accuracy: 1.00, Precision: 1.00, Recall: 0.93, F1-Score: 0.97


In [9]:
import pandas as pd
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.tree import DecisionTreeClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.svm import SVC


file_path = '/Users/sandhyavenkataramaiah/Downloads/ai4i2020.csv'
data = pd.read_csv(file_path)


data_prepared = data.drop(['UDI', 'Product ID'], axis=1)  
X = data_prepared.drop('Machine failure', axis=1)
y = data_prepared['Machine failure']


categorical_cols = [cname for cname in X.columns if X[cname].dtype == "object"]
numerical_cols = [cname for cname in X.columns if X[cname].dtype in ['int64', 'float64']]


preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_cols)
    ])


X = preprocessor.fit_transform(X)


n_folds = 5


models = {
    "Decision Tree": DecisionTreeClassifier(),
    "Neural Network (MLP)": MLPClassifier(max_iter=1000),
    "Support Vector Machine (SVM)": SVC()
}


def perform_cross_validation(model, X, y, n_folds):
    scores = cross_val_score(model, X, y, cv=n_folds, scoring='accuracy')
    print(f"Model: {model.__class__.__name__}, Accuracy: {scores.mean():.2f}, Standard Deviation: {scores.std():.2f}")


for model in models.values():
    perform_cross_validation(model, X, y, n_folds)

Model: DecisionTreeClassifier, Accuracy: 0.93, Standard Deviation: 0.14
Model: MLPClassifier, Accuracy: 1.00, Standard Deviation: 0.00
Model: SVC, Accuracy: 1.00, Standard Deviation: 0.00


In [12]:
import pandas as pd
from sklearn.model_selection import train_test_split, KFold
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.tree import DecisionTreeClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.base import clone
from sklearn.metrics import accuracy_score
import numpy as np

file_path = '/Users/sandhyavenkataramaiah/Downloads/ai4i2020.csv'
data = pd.read_csv(file_path)

data_prepared = data.drop(['UDI', 'Product ID'], axis=1)  # Drop irrelevant features
X = data_prepared.drop('Machine failure', axis=1)
y = data_prepared['Machine failure']

categorical_cols = [cname for cname in X.columns if X[cname].dtype == "object"]
numerical_cols = [cname for cname in X.columns if X[cname].dtype in ['int64', 'float64']]


preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_cols)
    ])


X = preprocessor.fit_transform(X)


X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0)


base_models = [
    DecisionTreeClassifier(),
    MLPClassifier(max_iter=1000),
    SVC(probability=True)
]


kf = KFold(n_splits=5, random_state=42, shuffle=True)


stacked_train = np.zeros((X_train.shape[0], len(base_models)))


for i, model in enumerate(base_models):
    for train_idx, valid_idx in kf.split(X_train):
    
        clone_model = clone(model)
        
   
        X_train_fold, y_train_fold = X_train[train_idx], y_train.iloc[train_idx]  
        
        clone_model.fit(X_train_fold, y_train_fold)
        stacked_train[valid_idx, i] = clone_model.predict(X_train[valid_idx])


meta_model = LogisticRegression()
meta_model.fit(stacked_train, y_train)


stacked_test = np.zeros((X_test.shape[0], len(base_models)))
for i, model in enumerate(base_models):
    model.fit(X_train, y_train)
    
  
    stacked_test[:, i] = model.predict(X_test)


final_predictions = meta_model.predict(stacked_test)


accuracy = accuracy_score(y_test, final_predictions)
print(f"Stacked Model Accuracy: {accuracy:.2f}")


Stacked Model Accuracy: 1.00
